In [18]:
import warnings
warnings.filterwarnings("ignore")

import time
import joblib
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [20]:
##DIRECTORIES##
ROOT = Path.cwd()

DATA_DIR = ROOT / "data"

PROCESSED_DIR = DATA_DIR / "processed"

MODEL_DIR = ROOT / "models"

OUTPUT_DIR = ROOT / "outputs"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [50]:
train = pd.read_csv(
    PROCESSED_DIR /
    "train.csv"
)


In [52]:
##drop date columns##
date_columns = [
    "application_date",
    "decision_timestamp",
    "monitoring_date"
]

train= train.drop(columns=date_columns, errors="ignore")

In [53]:
test = pd.read_csv(
    PROCESSED_DIR /
    "test.csv"
)
##drop date columns##
date_columns = [

    "application_date",

    "decision_timestamp",

    "monitoring_date"

]

test= test.drop(columns=date_columns, errors="ignore")

In [54]:
X_train = train.drop(
    columns=["loan_status"]
)

y_train = train["loan_status"]

X_test = test.drop(
    columns=["loan_status"]
)

y_test = test["loan_status"]

print(X_train.shape)
print(X_test.shape)

(8000, 39)
(2000, 39)


In [57]:
##Logistic Regression##
lr = Pipeline([
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

lr_parameters = {

    "model__C":[

        0.01,
        0.1,
        1,
        10

    ]

}

In [58]:
start = time.time()

lr_grid = GridSearchCV(

    lr,

    lr_parameters,

    cv=5,

    scoring="f1",

    n_jobs=-1

)

lr_grid.fit(

    X_train,

    y_train

)

lr_time = time.time() - start

joblib.dump(

    lr_grid.best_estimator_,

    MODEL_DIR /

    "logistic.pkl"

)

print(

    lr_grid.best_params_

)

print(

    lr_time

)

{'model__C': 1}
7.5770275592803955


In [59]:
##Random Forest##
rf = Pipeline([

    (

        "model",

        RandomForestClassifier(

            random_state=42

        )

    )

])

rf_parameters={

    "model__n_estimators":[

        100,

        200

    ],

    "model__max_depth":[

        5,

        10,

        None

    ]

}

In [60]:
start=time.time()

rf_grid=GridSearchCV(

    rf,

    rf_parameters,

    cv=5,

    scoring="f1",

    n_jobs=-1

)

rf_grid.fit(

    X_train,

    y_train

)

rf_time=time.time()-start

joblib.dump(

    rf_grid.best_estimator_,

    MODEL_DIR /

    "random_forest.pkl"

)

['C:\\Users\\oluwa\\models\\random_forest.pkl']

In [61]:
##XGBoost##
xgb = Pipeline([

    (

        "model",

        XGBClassifier(

            random_state=42,

            eval_metric="logloss",

            use_label_encoder=False

        )

    )

])

xgb_parameters={

    "model__n_estimators":[100,200],

    "model__max_depth":[3,6],

    "model__learning_rate":[0.05,0.1]

}

In [62]:
start=time.time()

xgb_grid=GridSearchCV(

    xgb,

    xgb_parameters,

    cv=5,

    scoring="f1",

    n_jobs=-1

)

xgb_grid.fit(

    X_train,

    y_train

)

xgb_time=time.time()-start

joblib.dump(

    xgb_grid.best_estimator_,

    MODEL_DIR /

    "xgboost.pkl"

)

['C:\\Users\\oluwa\\models\\xgboost.pkl']

In [63]:
##LightGBM##
lgb=LGBMClassifier(

    random_state=42

)

lgb_parameters={

    "n_estimators":[100,200],

    "learning_rate":[0.05,0.1],

    "max_depth":[5,10]

}

In [64]:
start=time.time()

lgb_grid=GridSearchCV(

    lgb,

    lgb_parameters,

    cv=5,

    scoring="f1",

    n_jobs=-1

)

lgb_grid.fit(

    X_train,

    y_train

)

lgb_time=time.time()-start

joblib.dump(

    lgb_grid.best_estimator_,

    MODEL_DIR /

    "lightgbm.pkl"

)

[LightGBM] [Info] Number of positive: 6594, number of negative: 1406
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002308 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3845
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.824250 -> initscore=1.545411
[LightGBM] [Info] Start training from score 1.545411
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

['C:\\Users\\oluwa\\models\\lightgbm.pkl']

In [66]:
##CatBoost##
cat = CatBoostClassifier(

    verbose=0,

    random_state=42

)

cat_parameters={

    "depth":[4,6],

    "iterations":[100,200],

    "learning_rate":[0.05,0.1]

}

In [67]:
start=time.time()

cat_grid=GridSearchCV(

    cat,

    cat_parameters,

    cv=5,

    scoring="f1",

    n_jobs=-1

)

cat_grid.fit(

    X_train,

    y_train

)

cat_time=time.time()-start

joblib.dump(

    cat_grid.best_estimator_,

    MODEL_DIR /

    "catboost.pkl"

)

['C:\\Users\\oluwa\\models\\catboost.pkl']

In [68]:
##Save Training Summary##
summary=pd.DataFrame({

    "Model":[

        "Logistic Regression",

        "Random Forest",

        "XGBoost",

        "LightGBM",

        "CatBoost"

    ],

    "Training Time (s)":[

        lr_time,

        rf_time,

        xgb_time,

        lgb_time,

        cat_time

    ],

    "Best Parameters":[

        str(lr_grid.best_params_),

        str(rf_grid.best_params_),

        str(xgb_grid.best_params_),

        str(lgb_grid.best_params_),

        str(cat_grid.best_params_)

    ]

})

summary

,Model,Training Time (s),Best Parameters
0,Logistic Regression,7.577028,{'model__C': 1}
1,Random Forest,22.608702,"{'model__max_depth': 5, 'model__n_estimators':..."
2,XGBoost,10.466386,"{'model__learning_rate': 0.05, 'model__max_dep..."
3,LightGBM,40.063299,"{'learning_rate': 0.05, 'max_depth': 5, 'n_est..."
4,CatBoost,28.399354,"{'depth': 6, 'iterations': 100, 'learning_rate..."


In [69]:
summary.to_csv(

    OUTPUT_DIR /

    "training_summary.csv",

    index=False

)

print("="*70)
print("MODEL TRAINING COMPLETE")
print("="*70)

print(summary)

MODEL TRAINING COMPLETE
                 Model  Training Time (s)  \
0  Logistic Regression           7.577028   
1        Random Forest          22.608702   
2              XGBoost          10.466386   
3             LightGBM          40.063299   
4             CatBoost          28.399354   

                                     Best Parameters  
0                                    {'model__C': 1}  
1  {'model__max_depth': 5, 'model__n_estimators':...  
2  {'model__learning_rate': 0.05, 'model__max_dep...  
3  {'learning_rate': 0.05, 'max_depth': 5, 'n_est...  
4  {'depth': 6, 'iterations': 100, 'learning_rate...  
